# EDA Notebook 5 — Course Catalog Bronze
**Source**: `mini_project_grp6.bronze.course_catalog_bronze`  
**Format**: XLSX/CSV | ~2,000 courses | Static reference data  
**Goal**: Duplicate check, rating distribution, active vs inactive courses, instructor load

In [0]:
from pyspark.sql import functions as F

TABLE = "mini_project_grp6.bronze.course_catalog_bronze"
df    = spark.table(TABLE)
total = df.count()

print(f"Total rows: {total:,}")
df.printSchema()

In [0]:
# ============================================================
# CELL 2 — Basic Stats & Null Audit
# ============================================================

df.describe().show()

null_df = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).toPandas().T.reset_index()
null_df.columns = ["column", "null_count"]
print(null_df.to_string(index=False))

In [0]:
# ============================================================
# CELL 3 — Duplicate course_id check (DQ Rule)
# ============================================================

dup_count = total - df.select("course_id").distinct().count()
print(f"Duplicate course_ids: {dup_count}")

if dup_count > 0:
    display(
        df.groupBy("course_id")
          .count()
          .filter(F.col("count") > 1)
          .orderBy("count", ascending=False)
    )

In [0]:
# ============================================================
# CELL 4 — avg_rating Validation (0–5 rule)
# ============================================================

invalid_rating = df.filter(
    (F.col("avg_rating") < 0) | (F.col("avg_rating") > 5)
)
print(f"Records with avg_rating outside [0,5]: {invalid_rating.count():,}")

print("\navg_rating distribution:")
df.select("avg_rating").describe().show()

display(
    df.withColumn("rating_bucket",
        F.when(F.col("avg_rating") < 2, "< 2.0 (Poor)")
         .when(F.col("avg_rating") < 3, "2.0–2.9")
         .when(F.col("avg_rating") < 4, "3.0–3.9")
         .when(F.col("avg_rating") < 4.5, "4.0–4.4")
         .otherwise("4.5+ (Excellent)")
    )
    .groupBy("rating_bucket")
    .count()
    .orderBy("rating_bucket")
)

In [0]:
# ============================================================
# CELL 5 — Active vs Inactive Courses
# ============================================================

display(
    df.groupBy("is_active")
      .agg(
          F.count("*").alias("count"),
          F.round(F.avg("avg_rating"), 2).alias("avg_rating"),
          F.round(F.avg("duration_hours"), 2).alias("avg_duration_hrs")
      )
)

In [0]:
# ============================================================
# CELL 6 — Subject Area & Difficulty Distribution
# ============================================================

print("=== SUBJECT AREA ===")
display(
    df.groupBy("subject_area")
      .count()
      .orderBy("count", ascending=False)
)

print("=== DIFFICULTY LEVEL ===")
display(
    df.groupBy("difficulty_level")
      .agg(
          F.count("*").alias("count"),
          F.round(F.avg("avg_rating"), 2).alias("avg_rating"),
          F.round(F.avg("price_inr"), 2).alias("avg_price_inr")
      )
      .orderBy("difficulty_level")
)

In [0]:
# ============================================================
# CELL 7 — Instructor Load (courses per instructor)
# ============================================================

display(
    df.groupBy("instructor_id", "instructor_name")
      .agg(
          F.count("*").alias("course_count"),
          F.round(F.avg("avg_rating"), 2).alias("avg_rating")
      )
      .orderBy("course_count", ascending=False)
      .limit(20)
)

In [0]:
# ============================================================
# CELL 8 — Course Content Depth
# ============================================================

df.selectExpr(
    "avg(total_modules)  as avg_modules",
    "avg(total_videos)   as avg_videos",
    "avg(total_quizzes)  as avg_quizzes",
    "avg(duration_hours) as avg_hours",
    "max(total_modules)  as max_modules",
    "min(total_modules)  as min_modules"
).show()